# ÉTAPE 4 — Feature Learning Automatique

Ce notebook implémente l'apprentissage automatique de caractéristiques latentes et temporelles (Feature Learning) pour enrichir notre dataset avant la modélisation finale.

## Trois techniques complémentaires sont utilisées :
1. **PCA (Analyse en Composantes Principales)** : Réduction dimensionnelle linéaire sur les 6 caractéristiques de température fortement corrélées (génère 3 composantes PCA).
2. **Auto-encodeur MLP (Keras)** : Apprentissage non linéaire de représentations compactes (génère un vecteur de goulot d'étranglement de 16 dimensions).
3. **Réseau LSTM Bidirectionnel / Unidirectionnel (Keras)** : Modélisation des dépendances temporelles sur des fenêtres glissantes de 14 jours par ville pour extraire des embeddings séquentiels de 64 dimensions.

## 🛡️ Prévention du Data Leakage :
* Le **StandardScaler** est entraîné uniquement sur le Train (`2009-2022`).
* La **PCA**, l'**Auto-encodeur** et le **LSTM** sont entraînés uniquement sur le Train (`2009-2022`) puis appliqués en inférence sur le reste du dataset.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Fixer les graines pour la reproductibilité
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

input_path = "../data/tunisie_meteo_features.csv"
output_path = "../data/tunisie_meteo_final.csv"
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

df = pd.read_csv(input_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by=["ville", "date"]).reset_index(drop=True)

# Masques des splits
train_mask = df["date"].dt.year <= 2022
val_mask = df["date"].dt.year == 2023
test_mask = df["date"].dt.year >= 2024

## 1. Normalisation StandardScaler

In [ ]:
core_weather_cols = [
    "temp_max", "temp_min", "temp_mean", "precipitation", "pluie", "neige_cm",
    "heures_pluie", "vent_max", "rafales_max", "vent_direction", "rayonnement",
    "evapotranspiration", "ensoleillement_h", "duree_jour_h", "humidite_max",
    "humidite_min", "humidite_mean", "rosee_max", "rosee_min", "ressenti_max",
    "ressenti_min", "pression_max", "pression_min", "pression_mean", "nuages_pct",
    "temp_sol", "humidite_sol", "amplitude_temp", "stress_hydrique", "ratio_ensoleillement"
]
core_weather_cols = [c for c in core_weather_cols if c in df.columns]

scaler = StandardScaler()
df_scaled = df.copy()
scaler.fit(df_scaled.loc[train_mask, core_weather_cols])
df_scaled[core_weather_cols] = scaler.transform(df_scaled[core_weather_cols])

## 2. Analyse en Composantes Principales (PCA)

Nous appliquons une PCA à 3 composantes sur les caractéristiques de température fortement corrélées.

In [ ]:
temp_cols = ["temp_max", "temp_min", "temp_mean", "amplitude_temp", "rosee_max", "rosee_min"]
temp_cols = [c for c in temp_cols if c in core_weather_cols]

pca = PCA(n_components=3, random_state=42)
pca.fit(df_scaled.loc[train_mask, temp_cols])

pca_features = pca.transform(df_scaled[temp_cols])
for i in range(3):
    df[f"pca_{i+1}"] = pca_features[:, i]

print(f"Variance expliquée par composante : {pca.explained_variance_ratio_}")
print(f"Cumul de variance expliquée : {sum(pca.explained_variance_ratio_):.4f}")

## 3. Auto-encodeur Keras (16 Dimensions Latentes)

In [ ]:
input_dim = len(core_weather_cols)

input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
encoded = Dense(32, activation='relu')(encoded)
bottleneck = Dense(16, name='bottleneck', activation='relu')(encoded)
decoded = Dense(32, activation='relu')(bottleneck)
decoded = Dense(64, activation='relu')(decoded)
output_layer = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
encoder = Model(inputs=input_layer, outputs=bottleneck)

autoencoder.compile(optimizer='adam', loss='mse')

X_train_ae = df_scaled.loc[train_mask, core_weather_cols].values
X_val_ae = df_scaled.loc[val_mask, core_weather_cols].values

early_stop_ae = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Entraînement de l'auto-encodeur...")
autoencoder.fit(
    X_train_ae, X_train_ae,
    validation_data=(X_val_ae, X_val_ae),
    epochs=30,
    batch_size=128,
    callbacks=[early_stop_ae],
    verbose=1
)

# Extraction et sauvegarde des features de l'auto-encodeur
ae_features = encoder.predict(df_scaled[core_weather_cols].values)
for i in range(16):
    df[f"ae_{i+1}"] = ae_features[:, i]

autoencoder.save(os.path.join(models_dir, "autoencoder_model.h5"))

## 4. Réseau LSTM & Embeddings Temporels (14 jours)

In [ ]:
time_steps = 14
X_seq = []
y_seq = []
weather_data_scaled = df_scaled[core_weather_cols].values
targets = df["pluie_demain_bin"].values
villes = df["ville"].values

print("Création des fenêtres glissantes de 14 jours par ville...")
for idx in range(len(df)):
    current_ville = villes[idx]
    start_idx = max(0, idx - time_steps + 1)
    city_start_idx = idx
    
    while city_start_idx > start_idx:
        if villes[city_start_idx - 1] != current_ville:
            break
        city_start_idx -= 1
        
    seq_slice = weather_data_scaled[city_start_idx : idx + 1]
    
    if len(seq_slice) < time_steps:
        pad_len = time_steps - len(seq_slice)
        pad_block = np.zeros((pad_len, input_dim))
        seq_slice = np.vstack([pad_block, seq_slice])
        
    X_seq.append(seq_slice)
    y_seq.append(targets[idx])
    
X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

X_train_lstm = X_seq[train_mask]
y_train_lstm = y_seq[train_mask]
X_val_lstm = X_seq[val_mask]
y_val_lstm = y_seq[val_mask]

In [ ]:
# Architecture du modèle LSTM
inputs_lstm = Input(shape=(time_steps, input_dim))
x = LSTM(64, return_sequences=True, dropout=0.3)(inputs_lstm)
lstm_out = LSTM(64, return_sequences=False, dropout=0.3)(x)
outputs_lstm = Dense(1, activation='sigmoid')(lstm_out)

lstm_model = Model(inputs=inputs_lstm, outputs=outputs_lstm)
lstm_extractor = Model(inputs=inputs_lstm, outputs=lstm_out)

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Pondération pour équilibrer la classe minoritaire
class_weight_0 = 1.0
class_weight_1 = len(y_train_lstm[y_train_lstm == 0]) / len(y_train_lstm[y_train_lstm == 1])
class_weights = {0: class_weight_0, 1: class_weight_1}

early_stop_lstm = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Entraînement du modèle LSTM supervisé...")
lstm_model.fit(
    X_train_lstm, y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    epochs=15,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop_lstm],
    verbose=1
)

# Extraction des embeddings temporels
print("Extraction des embeddings temporels de 64 dimensions...")
lstm_features = lstm_extractor.predict(X_seq)
for i in range(64):
    df[f"lstm_{i+1}"] = lstm_features[:, i]

lstm_model.save(os.path.join(models_dir, "lstm_supervised_model.h5"))

## 5. Sauvegarde du Dataset Final Consolidé

In [ ]:
# Sauvegarde du fichier consolidé contenant 160 colonnes
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Sauvegarde réussie : {output_path}")
print(f"Dimensions finales : {df.shape[0]:,} lignes, {df.shape[1]} colonnes.")